# DIET Intent Classification Training Data Generator

**For LocalCat Voice Agent Intent Classification**

This notebook generates high-quality training data for DIET intent classification using LLMs (OpenAI GPT-4 or Google Gemini). The generated data is optimized for voice agent interactions and exported in Rasa training format.

## Features
- 🤖 **LLM-powered generation**: Uses GPT-4 or Gemini for diverse, natural examples
- 🗣️ **Voice-optimized**: Focuses on spoken language patterns
- 📊 **Quality validation**: Automatic data quality checks and filtering
- 📁 **Export ready**: Direct export to Rasa YAML format
- ⚡ **Fast generation**: Batch processing for efficiency

## Quick Start
1. Set your API keys in the configuration section
2. Choose your LLM provider (OpenAI or Google)
3. Customize intent definitions if needed
4. Run all cells to generate and export training data

## Setup & Installation

In [ ]:
# Install required packages
!pip install openai requests pyyaml pandas numpy scikit-learn
!pip install google-generativeai  # For Google Gemini
!pip install anthropic            # For Claude (optional)

import os
import json
import yaml
import random
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import time
import requests
from collections import defaultdict
import re

print("✅ Dependencies installed successfully!")

## Configuration

In [ ]:
# ====== API CONFIGURATION ======
# Choose your preferred LLM provider and set the corresponding API key

# Option 1: OpenAI via OpenRouter (Recommended)
USE_OPENROUTER = True
OPENROUTER_API_KEY = ""  # Your OpenRouter API key
OPENROUTER_MODEL = "openai/gpt-4o-mini"  # Cost-effective model

# Option 2: Direct OpenAI
USE_OPENAI_DIRECT = False
OPENAI_API_KEY = ""  # Your OpenAI API key
OPENAI_MODEL = "gpt-4o-mini"

# Option 3: Google Gemini
USE_GOOGLE_AI = False
GOOGLE_AI_API_KEY = ""  # Your Google AI API key
GOOGLE_MODEL = "gemini-1.5-flash"

# ====== GENERATION CONFIGURATION ======
EXAMPLES_PER_INTENT = 25      # Number of examples to generate per intent
BATCH_SIZE = 5                # Examples per API call (for efficiency)
MAX_RETRIES = 3               # Retry failed API calls
TEMPERATURE = 0.9             # Higher for more diverse examples

# ====== QUALITY CONTROL ======
MIN_EXAMPLE_LENGTH = 3        # Minimum words per example
MAX_EXAMPLE_LENGTH = 20       # Maximum words per example
DUPLICATE_THRESHOLD = 0.8     # Similarity threshold for deduplication

# ====== OUTPUT CONFIGURATION ======
OUTPUT_FILENAME = "nlu_training_data.yml"
INCLUDE_VALIDATION_SPLIT = True
VALIDATION_RATIO = 0.2        # 20% for validation

print("✅ Configuration set!")
print(f"Provider: {'OpenRouter' if USE_OPENROUTER else 'OpenAI Direct' if USE_OPENAI_DIRECT else 'Google AI'}")
print(f"Examples per intent: {EXAMPLES_PER_INTENT}")
print(f"Total intents: {len([1])}")

## Intent Definitions

In [ ]:
# Define intents for voice agent with detailed descriptions and examples
INTENT_DEFINITIONS = {
    "remember_fact": {
        "description": "User wants the AI to store/remember information about themselves, their preferences, or facts",
        "voice_context": "Spoken requests to save personal information",
        "seed_examples": [
            "Remember that I like coffee",
            "Save this information please",
            "Don't forget I work at Google",
            "Keep in mind I'm allergic to peanuts"
        ],
        "variations": [
            "remember", "save", "store", "keep in mind", "don't forget", 
            "note that", "write down", "make a note", "keep track"
        ]
    },
    
    "recall_query": {
        "description": "User asks the AI to recall previously stored information",
        "voice_context": "Questions about stored personal information or memories",
        "seed_examples": [
            "What did I tell you about my job?",
            "Do you remember my favorite food?",
            "Remind me about my meeting",
            "What do you know about my family?"
        ],
        "variations": [
            "what did I", "do you remember", "remind me", "what do you know",
            "recall", "tell me about", "what information", "bring up"
        ]
    },
    
    "general_chat": {
        "description": "Casual conversation, small talk, or general questions not requiring memory operations",
        "voice_context": "Natural conversation, greetings, jokes, opinions, weather talk",
        "seed_examples": [
            "How are you doing today?",
            "Tell me a joke",
            "What's your favorite color?",
            "That's interesting"
        ],
        "variations": [
            "how are you", "tell me", "what's your", "that's", "I think",
            "you're", "this is", "pretty", "really", "kind of"
        ]
    },
    
    "forget_request": {
        "description": "User wants the AI to delete or forget previously stored information",
        "voice_context": "Requests to remove or delete stored information",
        "seed_examples": [
            "Forget what I said about that",
            "Delete that information",
            "Don't remember that anymore",
            "Remove that from your memory"
        ],
        "variations": [
            "forget", "delete", "remove", "don't remember", "erase",
            "get rid of", "clear", "wipe", "discard"
        ]
    },
    
    "clarification": {
        "description": "User asks for explanation, clarification, or doesn't understand something",
        "voice_context": "Confusion, asking for more details or explanation",
        "seed_examples": [
            "What do you mean?",
            "Can you explain that?",
            "I don't understand",
            "Could you clarify?"
        ],
        "variations": [
            "what do you mean", "explain", "clarify", "I don't understand",
            "confused", "not sure", "can you", "help me understand"
        ]
    },
    
    "correction": {
        "description": "User corrects or updates previously stated information",
        "voice_context": "Fixing mistakes, updating information, contradicting previous statements",
        "seed_examples": [
            "No, that's wrong",
            "Actually, it's different",
            "Let me correct that",
            "I misspoke earlier"
        ],
        "variations": [
            "no", "actually", "correct", "wrong", "mistake", "fix",
            "change", "update", "meant to say", "I misspoke"
        ]
    },
    
    "greeting": {
        "description": "Initial greetings and conversation starters",
        "voice_context": "Beginning of conversation, hellos, morning greetings",
        "seed_examples": [
            "Hello",
            "Hi there",
            "Good morning",
            "Hey"
        ],
        "variations": [
            "hello", "hi", "hey", "good morning", "good afternoon",
            "what's up", "how's it going", "greetings"
        ]
    },
    
    "goodbye": {
        "description": "Ending conversation, farewells, signing off",
        "voice_context": "End of conversation, goodbye messages",
        "seed_examples": [
            "Goodbye",
            "See you later",
            "Bye",
            "Talk to you soon"
        ],
        "variations": [
            "goodbye", "bye", "see you", "talk to you", "catch you",
            "have a good", "until next time", "farewell"
        ]
    },
    
    "affirmation": {
        "description": "Agreement, confirmation, positive responses",
        "voice_context": "Agreeing with AI responses, confirming information",
        "seed_examples": [
            "Yes",
            "That's right",
            "Correct",
            "Exactly"
        ],
        "variations": [
            "yes", "right", "correct", "exactly", "absolutely",
            "you got it", "that's it", "perfect", "agreed"
        ]
    },
    
    "negation": {
        "description": "Disagreement, denial, negative responses",
        "voice_context": "Disagreeing with AI, correcting misunderstandings",
        "seed_examples": [
            "No",
            "That's wrong",
            "Incorrect",
            "Not really"
        ],
        "variations": [
            "no", "wrong", "incorrect", "not really", "nope",
            "I disagree", "that's not right", "false"
        ]
    }
}

print(f"✅ Defined {len(INTENT_DEFINITIONS)} intents:")
for intent, definition in INTENT_DEFINITIONS.items():
    print(f"  • {intent}: {definition['description'][:60]}...")

## LLM Client Classes

In [ ]:
import openai
try:
    import google.generativeai as genai
    GOOGLE_AVAILABLE = True
except ImportError:
    GOOGLE_AVAILABLE = False

@dataclass
class GenerationRequest:
    intent: str
    description: str
    examples: List[str]
    count: int
    
class BaseLLMClient:
    """Base class for LLM clients"""
    
    def __init__(self, temperature: float = 0.9):
        self.temperature = temperature
    
    async def generate_examples(self, request: GenerationRequest) -> List[str]:
        raise NotImplementedError
    
    def create_prompt(self, request: GenerationRequest) -> str:
        """Create generation prompt"""
        prompt = f"""Generate {request.count} diverse training examples for the intent '{request.intent}'.

Intent Description: {request.description}

Context: These are for a voice AI assistant. Examples should be:
- Natural spoken language (contractions, casual tone)
- Varied in length (2-15 words)
- Different phrasings of the same intent
- Include questions, statements, and commands as appropriate

Seed Examples:
{chr(10).join(f'- {ex}' for ex in request.examples)}

Generate {request.count} NEW examples (do not repeat seed examples):
Format as a simple list, one example per line, no numbers or bullets."""
        return prompt


class OpenRouterClient(BaseLLMClient):
    """OpenRouter client for cost-effective access to multiple models"""
    
    def __init__(self, api_key: str, model: str = "openai/gpt-4o-mini", temperature: float = 0.9):
        super().__init__(temperature)
        self.api_key = api_key
        self.model = model
        self.base_url = "https://openrouter.ai/api/v1"
    
    async def generate_examples(self, request: GenerationRequest) -> List[str]:
        prompt = self.create_prompt(request)
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://localcat-voice-agent.com",
            "X-Title": "LocalCat Voice Agent Training Data Generator"
        }
        
        payload = {
            "model": self.model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": self.temperature,
            "max_tokens": 500
        }
        
        response = requests.post(
            f"{self.base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )
        
        if response.status_code != 200:
            raise Exception(f"OpenRouter API error: {response.status_code} - {response.text}")
        
        result = response.json()
        content = result['choices'][0]['message']['content']
        
        # Parse examples from response
        examples = []
        for line in content.strip().split('\n'):
            line = line.strip()
            if line and not line.startswith('#') and not line.startswith('**'):
                # Remove bullets, numbers, etc.
                line = re.sub(r'^[\d\-\*\•]+\.?\s*', '', line)
                if line:
                    examples.append(line)
        
        return examples[:request.count]


class OpenAIClient(BaseLLMClient):
    """Direct OpenAI client"""
    
    def __init__(self, api_key: str, model: str = "gpt-4o-mini", temperature: float = 0.9):
        super().__init__(temperature)
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model
    
    async def generate_examples(self, request: GenerationRequest) -> List[str]:
        prompt = self.create_prompt(request)
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=self.temperature,
            max_tokens=500
        )
        
        content = response.choices[0].message.content
        
        # Parse examples
        examples = []
        for line in content.strip().split('\n'):
            line = line.strip()
            if line and not line.startswith('#') and not line.startswith('**'):
                line = re.sub(r'^[\d\-\*\•]+\.?\s*', '', line)
                if line:
                    examples.append(line)
        
        return examples[:request.count]


class GoogleAIClient(BaseLLMClient):
    """Google Gemini client"""
    
    def __init__(self, api_key: str, model: str = "gemini-1.5-flash", temperature: float = 0.9):
        super().__init__(temperature)
        if not GOOGLE_AVAILABLE:
            raise ImportError("google-generativeai not installed")
        
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model)
    
    async def generate_examples(self, request: GenerationRequest) -> List[str]:
        prompt = self.create_prompt(request)
        
        generation_config = genai.types.GenerationConfig(
            temperature=self.temperature,
            max_output_tokens=500
        )
        
        response = self.model.generate_content(
            prompt,
            generation_config=generation_config
        )
        
        content = response.text
        
        # Parse examples
        examples = []
        for line in content.strip().split('\n'):
            line = line.strip()
            if line and not line.startswith('#') and not line.startswith('**'):
                line = re.sub(r'^[\d\-\*\•]+\.?\s*', '', line)
                if line:
                    examples.append(line)
        
        return examples[:request.count]


def create_llm_client() -> BaseLLMClient:
    """Factory function to create the configured LLM client"""
    if USE_OPENROUTER and OPENROUTER_API_KEY:
        return OpenRouterClient(OPENROUTER_API_KEY, OPENROUTER_MODEL, TEMPERATURE)
    elif USE_OPENAI_DIRECT and OPENAI_API_KEY:
        return OpenAIClient(OPENAI_API_KEY, OPENAI_MODEL, TEMPERATURE)
    elif USE_GOOGLE_AI and GOOGLE_AI_API_KEY:
        return GoogleAIClient(GOOGLE_AI_API_KEY, GOOGLE_MODEL, TEMPERATURE)
    else:
        raise ValueError("No valid LLM configuration found. Please set API keys.")

print("✅ LLM clients configured!")

## Data Generation Engine

In [ ]:
import asyncio
from typing import Tuple
import difflib

class TrainingDataGenerator:
    """Main class for generating training data"""
    
    def __init__(self, llm_client: BaseLLMClient):
        self.llm_client = llm_client
        self.generated_data = defaultdict(list)
        self.generation_stats = {
            'total_requests': 0,
            'successful_requests': 0,
            'failed_requests': 0,
            'duplicates_removed': 0,
            'examples_generated': 0
        }
    
    def filter_examples(self, examples: List[str], intent: str) -> List[str]:
        """Filter and validate generated examples"""
        filtered = []
        existing_examples = self.generated_data[intent]
        
        for example in examples:
            example = example.strip()
            
            # Basic validation
            word_count = len(example.split())
            if word_count < MIN_EXAMPLE_LENGTH or word_count > MAX_EXAMPLE_LENGTH:
                continue
            
            # Remove examples that are too similar to existing ones
            is_duplicate = False
            for existing in existing_examples + filtered:
                similarity = difflib.SequenceMatcher(None, example.lower(), existing.lower()).ratio()
                if similarity > DUPLICATE_THRESHOLD:
                    is_duplicate = True
                    self.generation_stats['duplicates_removed'] += 1
                    break
            
            if not is_duplicate:
                filtered.append(example)
        
        return filtered
    
    async def generate_for_intent(self, intent: str, definition: Dict[str, Any]) -> int:
        """Generate examples for a single intent"""
        print(f"🎯 Generating examples for '{intent}'...")
        
        examples_needed = EXAMPLES_PER_INTENT
        examples_generated = 0
        attempts = 0
        max_attempts = 10
        
        # Start with seed examples
        seed_examples = definition['seed_examples']
        self.generated_data[intent].extend(seed_examples)
        examples_generated += len(seed_examples)
        examples_needed -= len(seed_examples)
        
        while examples_needed > 0 and attempts < max_attempts:
            attempts += 1
            batch_size = min(BATCH_SIZE, examples_needed)
            
            try:
                request = GenerationRequest(
                    intent=intent,
                    description=definition['description'],
                    examples=seed_examples,
                    count=batch_size
                )
                
                self.generation_stats['total_requests'] += 1
                
                # Generate examples
                raw_examples = await self.llm_client.generate_examples(request)
                
                # Filter and validate
                filtered_examples = self.filter_examples(raw_examples, intent)
                
                if filtered_examples:
                    self.generated_data[intent].extend(filtered_examples)
                    examples_generated += len(filtered_examples)
                    examples_needed -= len(filtered_examples)
                    self.generation_stats['successful_requests'] += 1
                    
                    print(f"   ✓ Generated {len(filtered_examples)} examples (batch {attempts})")
                else:
                    print(f"   ⚠ No valid examples in batch {attempts}")
                
                # Rate limiting
                await asyncio.sleep(0.5)
                
            except Exception as e:
                print(f"   ❌ Error in batch {attempts}: {str(e)}")
                self.generation_stats['failed_requests'] += 1
                await asyncio.sleep(2)  # Longer delay on error
        
        final_count = len(self.generated_data[intent])
        self.generation_stats['examples_generated'] += final_count
        
        print(f"   🎉 Completed '{intent}': {final_count} total examples")
        return final_count
    
    async def generate_all(self) -> Dict[str, List[str]]:
        """Generate examples for all intents"""
        print(f"🚀 Starting generation for {len(INTENT_DEFINITIONS)} intents...\n")
        
        start_time = time.time()
        
        # Generate for each intent
        for intent, definition in INTENT_DEFINITIONS.items():
            await self.generate_for_intent(intent, definition)
            print()  # Empty line for readability
        
        elapsed_time = time.time() - start_time
        
        # Print summary
        print("="*60)
        print("📊 GENERATION SUMMARY")
        print("="*60)
        print(f"⏱️  Total time: {elapsed_time:.1f} seconds")
        print(f"🎯 Intents processed: {len(self.generated_data)}")
        print(f"📝 Total examples: {sum(len(examples) for examples in self.generated_data.values())}")
        print(f"✅ Successful API calls: {self.generation_stats['successful_requests']}")
        print(f"❌ Failed API calls: {self.generation_stats['failed_requests']}")
        print(f"🗑️  Duplicates removed: {self.generation_stats['duplicates_removed']}")
        print()
        
        # Per-intent breakdown
        print("📋 EXAMPLES PER INTENT:")
        for intent, examples in self.generated_data.items():
            print(f"   {intent}: {len(examples)} examples")
        
        return dict(self.generated_data)

print("✅ Data generation engine ready!")

## Export Utilities

In [ ]:
from sklearn.model_selection import train_test_split

class DataExporter:
    """Export generated data in various formats"""
    
    def __init__(self, data: Dict[str, List[str]]):
        self.data = data
    
    def create_train_test_split(self) -> Tuple[Dict[str, List[str]], Dict[str, List[str]]]:
        """Split data into training and validation sets"""
        train_data = {}
        test_data = {}
        
        for intent, examples in self.data.items():
            if len(examples) > 4:  # Only split if we have enough examples
                train_examples, test_examples = train_test_split(
                    examples, 
                    test_size=VALIDATION_RATIO, 
                    random_state=42
                )
                train_data[intent] = train_examples
                test_data[intent] = test_examples
            else:
                # Put all examples in training if too few
                train_data[intent] = examples
                test_data[intent] = []
        
        return train_data, test_data
    
    def export_rasa_yaml(self, filename: str = None, include_validation: bool = True) -> str:
        """Export data in Rasa YAML format"""
        if filename is None:
            filename = OUTPUT_FILENAME
        
        if include_validation and INCLUDE_VALIDATION_SPLIT:
            train_data, test_data = self.create_train_test_split()
            data_to_export = train_data
        else:
            data_to_export = self.data
            test_data = None
        
        # Create YAML structure
        nlu_data = {
            'version': '3.1',
            'nlu': []
        }
        
        for intent, examples in data_to_export.items():
            intent_block = {
                'intent': intent,
                'examples': '\n'.join(f'- {example}' for example in examples)
            }
            nlu_data['nlu'].append(intent_block)
        
        # Export training data
        yaml_content = yaml.dump(nlu_data, default_flow_style=False, allow_unicode=True)
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(yaml_content)
        
        print(f"✅ Training data exported to: {filename}")
        
        # Export validation data if split
        if test_data and any(test_data.values()):
            test_filename = filename.replace('.yml', '_validation.yml')
            test_nlu_data = {
                'version': '3.1',
                'nlu': []
            }
            
            for intent, examples in test_data.items():
                if examples:  # Only include intents with validation examples
                    intent_block = {
                        'intent': intent,
                        'examples': '\n'.join(f'- {example}' for example in examples)
                    }
                    test_nlu_data['nlu'].append(intent_block)
            
            test_yaml_content = yaml.dump(test_nlu_data, default_flow_style=False, allow_unicode=True)
            
            with open(test_filename, 'w', encoding='utf-8') as f:
                f.write(test_yaml_content)
            
            print(f"✅ Validation data exported to: {test_filename}")
        
        return filename
    
    def export_json(self, filename: str = None) -> str:
        """Export data as JSON for analysis"""
        if filename is None:
            filename = OUTPUT_FILENAME.replace('.yml', '.json')
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.data, f, indent=2, ensure_ascii=False)
        
        print(f"✅ JSON data exported to: {filename}")
        return filename
    
    def export_csv(self, filename: str = None) -> str:
        """Export data as CSV for analysis"""
        if filename is None:
            filename = OUTPUT_FILENAME.replace('.yml', '.csv')
        
        # Flatten data for CSV
        rows = []
        for intent, examples in self.data.items():
            for example in examples:
                rows.append({
                    'intent': intent,
                    'text': example,
                    'word_count': len(example.split())
                })
        
        df = pd.DataFrame(rows)
        df.to_csv(filename, index=False, encoding='utf-8')
        
        print(f"✅ CSV data exported to: {filename}")
        return filename
    
    def print_sample_data(self, samples_per_intent: int = 3):
        """Print sample examples for verification"""
        print("📋 SAMPLE GENERATED DATA:")
        print("="*60)
        
        for intent, examples in self.data.items():
            print(f"\n🎯 {intent.upper()} ({len(examples)} total examples):")
            for i, example in enumerate(examples[:samples_per_intent]):
                print(f"   {i+1}. {example}")
            if len(examples) > samples_per_intent:
                print(f"   ... and {len(examples) - samples_per_intent} more")

print("✅ Export utilities ready!")

## Generate Training Data

**🚀 Run this cell to generate your training data!**

Make sure you've:
1. ✅ Set your API keys in the configuration section
2. ✅ Reviewed the intent definitions above
3. ✅ Customized generation parameters if needed

In [ ]:
# Check configuration
if not (OPENROUTER_API_KEY or OPENAI_API_KEY or GOOGLE_AI_API_KEY):
    print("❌ ERROR: No API key configured!")
    print("Please set your API key in the configuration section above.")
else:
    print("🔑 API configuration validated!")
    
    # Create LLM client
    try:
        llm_client = create_llm_client()
        print(f"✅ LLM client created successfully!")
    except Exception as e:
        print(f"❌ Failed to create LLM client: {e}")
        raise
    
    # Create generator
    generator = TrainingDataGenerator(llm_client)
    
    # Generate data
    print("\n🎬 Starting data generation...\n")
    generated_data = await generator.generate_all()
    
    # Create exporter and show samples
    exporter = DataExporter(generated_data)
    exporter.print_sample_data()
    
    print("\n🎉 Data generation completed successfully!")

## Export Training Data

Export your generated data in multiple formats for use with DIET/Rasa training.

In [ ]:
# Export in all formats
print("📦 Exporting training data in multiple formats...\n")

# Primary export: Rasa YAML format
yaml_file = exporter.export_rasa_yaml()

# Additional formats for analysis
json_file = exporter.export_json()
csv_file = exporter.export_csv()

print("\n📁 FILES CREATED:")
print(f"   🎯 Training data (Rasa): {yaml_file}")
if INCLUDE_VALIDATION_SPLIT:
    validation_file = yaml_file.replace('.yml', '_validation.yml')
    print(f"   📊 Validation data: {validation_file}")
print(f"   📋 Analysis (JSON): {json_file}")
print(f"   📈 Analysis (CSV): {csv_file}")

print("\n✅ All exports completed!")
print("\n📚 NEXT STEPS:")
print("1. Download the YAML files to your LocalCat project")
print("2. Follow the implementation guide for DIET integration")
print("3. Train your DIET model with: `rasa train nlu`")
print("4. Test and iterate with real voice data")

## Data Quality Analysis

Analyze the generated training data for quality and balance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create DataFrame for analysis
analysis_data = []
for intent, examples in generated_data.items():
    for example in examples:
        analysis_data.append({
            'intent': intent,
            'text': example,
            'word_count': len(example.split()),
            'char_count': len(example),
            'has_question': '?' in example,
            'has_contraction': any(word in example.lower() for word in ["i'm", "you're", "don't", "won't", "can't", "it's"])
        })

df = pd.DataFrame(analysis_data)

# Basic statistics
print("📊 DATA QUALITY ANALYSIS")
print("="*50)
print(f"Total examples: {len(df)}")
print(f"Total intents: {df['intent'].nunique()}")
print(f"Average examples per intent: {len(df) / df['intent'].nunique():.1f}")
print(f"Average word count: {df['word_count'].mean():.1f}")
print(f"Average character count: {df['char_count'].mean():.1f}")
print(f"Percentage with questions: {df['has_question'].mean() * 100:.1f}%")
print(f"Percentage with contractions: {df['has_contraction'].mean() * 100:.1f}%")

# Intent distribution
print("\n📋 EXAMPLES PER INTENT:")
intent_counts = df['intent'].value_counts().sort_index()
for intent, count in intent_counts.items():
    print(f"   {intent}: {count} examples")

# Word count distribution
print("\n📏 WORD COUNT DISTRIBUTION:")
word_count_stats = df['word_count'].describe()
for stat, value in word_count_stats.items():
    print(f"   {stat}: {value:.1f}")

# Check for potential issues
print("\n⚠️  QUALITY CHECKS:")
short_examples = df[df['word_count'] < 3]
long_examples = df[df['word_count'] > 15]
print(f"   Examples too short (<3 words): {len(short_examples)}")
print(f"   Examples too long (>15 words): {len(long_examples)}")

# Find potential duplicates
duplicates = df[df.duplicated(subset=['text'], keep=False)]
print(f"   Exact duplicates: {len(duplicates)}")

if len(duplicates) > 0:
    print("\n🔍 DUPLICATE EXAMPLES:")
    for _, row in duplicates.iterrows():
        print(f"   {row['intent']}: {row['text']}")

# Plot distribution (if matplotlib is available)
try:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Intent distribution
    intent_counts.plot(kind='bar', ax=axes[0, 0], title='Examples per Intent')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Word count distribution
    df['word_count'].hist(bins=15, ax=axes[0, 1], title='Word Count Distribution')
    
    # Character count distribution
    df['char_count'].hist(bins=15, ax=axes[1, 0], title='Character Count Distribution')
    
    # Box plot of word counts by intent
    df.boxplot(column='word_count', by='intent', ax=axes[1, 1])
    axes[1, 1].set_title('Word Count by Intent')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("\n📊 Install matplotlib for visualizations: pip install matplotlib")

print("\n✅ Data quality analysis completed!")

## Model Training Recommendations

Based on your generated data, here are specific recommendations for training your DIET model.

In [ ]:
# Generate training recommendations based on data analysis
total_examples = len(df)
min_examples = intent_counts.min()
max_examples = intent_counts.max()
std_examples = intent_counts.std()

print("🎯 DIET TRAINING RECOMMENDATIONS")
print("="*50)

# Training configuration recommendations
if total_examples < 200:
    epochs = 150
    print("📊 Small dataset detected (<200 examples)")
elif total_examples < 500:
    epochs = 100
    print("📊 Medium dataset detected (200-500 examples)")
else:
    epochs = 80
    print("📊 Large dataset detected (>500 examples)")

print(f"\n⚙️  RECOMMENDED DIET CONFIG:")
print(f"   - epochs: {epochs}")
print(f"   - constrain_similarities: true")
print(f"   - model_confidence: softmax")
print(f"   - entity_recognition: false  # Intent only")

# Data balance analysis
if std_examples > (intent_counts.mean() * 0.3):
    print("\n⚠️  IMBALANCED DATA DETECTED")
    print("   Consider generating more examples for underrepresented intents:")
    underrepresented = intent_counts[intent_counts < (intent_counts.mean() * 0.8)]
    for intent, count in underrepresented.items():
        print(f"   - {intent}: {count} examples (add {int(intent_counts.mean() - count)} more)")
else:
    print("\n✅ Well-balanced dataset!")

# Performance expectations
print(f"\n🎯 EXPECTED PERFORMANCE:")
if min_examples >= 20:
    print("   - High accuracy expected (>90%)")
elif min_examples >= 10:
    print("   - Good accuracy expected (80-90%)")
else:
    print("   - Moderate accuracy expected (70-80%)")
    print("   - Consider generating more examples")

print(f"   - Training time: ~{epochs * len(intent_counts) * 0.1:.1f} seconds")
print(f"   - Model size: ~{total_examples * 0.001:.1f} MB")

# Integration recommendations
print(f"\n🔧 INTEGRATION SETTINGS:")
if min_examples >= 20:
    confidence_threshold = 0.8
elif min_examples >= 15:
    confidence_threshold = 0.7
else:
    confidence_threshold = 0.6

print(f"   - DIET_CONFIDENCE_THRESHOLD: {confidence_threshold}")
print(f"   - DIET_FALLBACK_TO_MEMORY: true")
print(f"   - Expected inference time: 10-20ms")

# Next steps
print(f"\n📚 NEXT STEPS:")
print(f"1. Copy generated YAML to your Rasa project")
print(f"2. Use recommended config.yml settings above")
print(f"3. Train with: `rasa train nlu --config config.yml`")
print(f"4. Test accuracy with validation set")
print(f"5. Deploy in LocalCat with confidence threshold {confidence_threshold}")
print(f"6. Monitor performance and retrain as needed")

# Generate config file content
config_content = f"""language: en
pipeline:
  - name: WhitespaceTokenizer
  - name: RegexFeaturizer
  - name: LexicalSyntacticFeaturizer
  - name: CountVectorsFeaturizer
  - name: CountVectorsFeaturizer
    analyzer: char_wb
    min_ngram: 1
    max_ngram: 4
  - name: DIETClassifier
    epochs: {epochs}
    constrain_similarities: true
    model_confidence: softmax
    entity_recognition: false
  - name: FallbackClassifier
    threshold: {confidence_threshold}
    ambiguity_threshold: 0.1

policies:
  - name: MemoizationPolicy
  - name: RulePolicy
"""

# Save recommended config
with open('recommended_config.yml', 'w') as f:
    f.write(config_content)

print(f"\n💾 Saved recommended configuration to: recommended_config.yml")
print(f"\n✅ Training recommendations completed!")

## Download Generated Files

Your training data and configuration files are ready! Download them to integrate with your LocalCat voice agent.

In [ ]:
import os
from google.colab import files

# List all generated files
generated_files = [
    OUTPUT_FILENAME,  # Training data
    OUTPUT_FILENAME.replace('.yml', '_validation.yml'),  # Validation data
    OUTPUT_FILENAME.replace('.yml', '.json'),  # JSON export
    OUTPUT_FILENAME.replace('.yml', '.csv'),   # CSV export
    'recommended_config.yml'  # DIET configuration
]

print("📁 GENERATED FILES:")
print("="*40)

existing_files = []
for filename in generated_files:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        print(f"✅ {filename} ({size:,} bytes)")
        existing_files.append(filename)
    else:
        print(f"❌ {filename} (not found)")

print(f"\n📦 Ready to download {len(existing_files)} files")

# Download all files
if existing_files:
    print("\n⬇️  Downloading files...")
    for filename in existing_files:
        try:
            files.download(filename)
            print(f"   ✅ Downloaded: {filename}")
        except Exception as e:
            print(f"   ❌ Failed to download {filename}: {e}")
    
    print("\n🎉 All files downloaded!")
    print("\n📋 INTEGRATION CHECKLIST:")
    print("   □ Copy YAML files to your Rasa project data/ directory")
    print("   □ Copy recommended_config.yml to your Rasa project root")
    print("   □ Run `rasa train nlu` to train your DIET model")
    print("   □ Follow the LocalCat integration guide")
    print("   □ Test intent classification with real voice data")
else:
    print("❌ No files to download. Please run the generation steps above.")

## 🎉 Congratulations!

You've successfully generated high-quality training data for DIET intent classification! Here's what you've accomplished:

### ✅ Generated Training Data
- **10 voice-optimized intents** for your LocalCat agent
- **25+ examples per intent** with natural language variations
- **Quality filtered** data with duplicate removal
- **Train/validation split** for proper model evaluation

### ✅ Ready-to-Use Files
- `nlu_training_data.yml` - Primary training data
- `nlu_training_data_validation.yml` - Validation set
- `recommended_config.yml` - Optimized DIET configuration
- Analysis files (JSON/CSV) for further exploration

### 🚀 Next Steps
1. **Train your model**: Use the generated data with Rasa
2. **Integrate with LocalCat**: Follow the implementation guide
3. **Test & iterate**: Refine based on real voice interactions
4. **Monitor performance**: Track intent classification accuracy

### 📚 Resources
- [DIET Implementation Guide](../docs/diet-intent-classification-guide.md)
- [Discovery Report](../backlog/drafts/diet-intent-classification-discovery.md)
- [Rasa DIET Documentation](https://rasa.com/docs/rasa/nlu/components/#dietclassifier)

---

**Happy voice agent building! 🤖🗣️**